# Gemma-2-9B-it on the first 10 TriviaQA questions, scored by the Qwen detector

Takes the detector of `nvthaai/sala-g14-logsumexp-layerpca512-4x128` (log-sum-exp pooling, per-layer PCA 256, 4 x 128 MLP,
trained on Qwen2.5-7B hidden states) and applies it to **Gemma-2-9B-it**:
1. Gemma answers the first 10 TriviaQA questions of the saved set (greedy, max 32 tokens, same short-answer instruction).
   Its last answer token's hidden state is kept for every layer: `(10, 43, 3584)`.
2. Layer count: pooling brings Gemma's 43 hidden states down to the 15 pooled layers the detector expects.
3. Hidden size: Gemma-2-9B is also 3584 wide, so Qwen's stored PCA matrices can be applied without error. The neurons of
   the two models are unrelated, so **expect the scores to be close to meaningless**; this run only shows whether the
   plumbing works. Ten examples cannot give a reliable AUROC, so only per-example results are shown.
4. As a control, the same 10 questions are scored with **Qwen's own saved hidden states**; these must reproduce the
   original run's probabilities.

**Attach as inputs:** (1) the features dataset `sala_qwen25_g14_6src_triviaqa_v1`, (2) the output of the notebook above
(Add Input -> Notebooks), (3) Gemma: Add Input -> Models -> google / gemma-2 / Transformers / gemma-2-9b-it
(accept the Gemma license on Kaggle first). Add the `GITHUB_TOKEN` secret, turn on GPU (T4 x2) and internet.

In [ ]:
# Clones the private repo using the GITHUB_TOKEN Kaggle secret (Add-ons -> Secrets -> attach it to this notebook).
# The token is passed to git only through an environment header, never in the command line or URL, so it cannot show
# up in tracebacks, logs or the saved clone.
import base64, os, subprocess, urllib.request
from kaggle_secrets import UserSecretsClient

REPO, REF = "brainardphilemon/hallucination-probes", "main"   # pin a commit hash for reproducible runs
DEST = "/kaggle/working/hallucination-probes"
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
env = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1",
       "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader", "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"}
subprocess.run(["rm", "-rf", DEST], check=True)
clone = subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git", DEST], env=env,
                       capture_output=True, text=True)
if clone.returncode:
    try:  # 401 = token invalid/expired; 404 = token cannot see this repo (wrong repository or missing Contents: read)
        request = urllib.request.Request(f"https://api.github.com/repos/{REPO}", headers={"Authorization": f"Bearer {token}"})
        status = urllib.request.urlopen(request).status
    except Exception as exc:
        status = getattr(exc, "code", type(exc).__name__)
    raise RuntimeError(f"git clone failed; GitHub API answered {status} for {REPO} with your token. 401: token invalid or "
                       "expired. 404: the token has no read access to this repository (create it with Resource owner "
                       "brainardphilemon, repository hallucination-probes, Contents: Read-only).") from None
del token, auth, env
subprocess.run(["git", "-C", DEST, "checkout", "-q", REF], check=True)
print(subprocess.run(["git", "-C", DEST, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r /kaggle/working/hallucination-probes/requirements-kaggle.txt
!pip install -q --no-deps -e /kaggle/working/hallucination-probes

In [ ]:
# ---- config -------------------------------------------------------------------------------
N = 10                                                     # first N TriviaQA questions
FEATURES_NAME = "sala_qwen25_g14_6src_triviaqa_v1"         # folder with generation/ and bleurt/
RUN_FOLDER = "sala_g14_6src_logsumexp_layerpca512_4x128"   # folder holding best_mlp.pt and summary.json

In [ ]:
import json
from pathlib import Path

def find_one(name, test):
    hits = [p for p in Path("/kaggle/input").rglob(name) if test(p)]
    if len(hits) != 1:
        raise RuntimeError(f"Expected exactly one {name} under /kaggle/input; found {hits}")
    return hits[0]

def is_gemma_9b(config_path):
    c = json.loads(config_path.read_text())
    return c.get("model_type") == "gemma2" and c.get("hidden_size") == 3584 and c.get("num_hidden_layers") == 42

FEATURES = find_one(FEATURES_NAME, lambda p: (p / "generation").is_dir())
RUN_DIR = find_one(RUN_FOLDER, lambda p: (p / "best_mlp.pt").is_file())
GEMMA = find_one("config.json", is_gemma_9b).parent
print(FEATURES, RUN_DIR, GEMMA, sep="\n")

In [ ]:
# First N TriviaQA questions of the saved set (the same ones Qwen answered), with their reference answers.
rows = [json.loads(l) for l in (FEATURES / "generation" / "triviaqa" / "rows.jsonl").read_text().splitlines()[:N]]
out = Path("/kaggle/working")
with (out / "questions.jsonl").open("w") as f:
    for r in rows:
        f.write(json.dumps({"id": r["id"], "input_text": r["input_text"], "ground_truth_answers": r["ground_truth_answers"]}) + "\n")
print(len(rows), "questions; first:", rows[0]["input_text"])

In [ ]:
# Gemma answers the questions and we keep its hidden states (4-bit, bf16 compute, eager attention for Gemma 2).
!hprobes generate --model "{GEMMA}" --questions /kaggle/working/questions.jsonl --limit {N} --output-dir /kaggle/working/gemma

In [ ]:
# Rebuild the detector: Qwen's saved MLP + its per-layer PCA refitted from the saved features (checked against the original run).
!hprobes import-legacy --run-dir "{RUN_DIR}" --artifact-dir "{FEATURES}" --output /kaggle/working/model.pt --device cuda

In [ ]:
# Gemma hidden states (43 layers x 3584) -> pooled to 15 layers -> Qwen's PCA -> MLP. BLEURT labels Gemma's answers vs the references.
!hprobes predict --model /kaggle/working/model.pt --states /kaggle/working/gemma/states.npy --records /kaggle/working/gemma/records.jsonl --bleurt --any-layers --device cuda --output /kaggle/working/gemma_predictions.csv

In [ ]:
# Control: the same questions with Qwen's own saved hidden states and answers.
import numpy as np
np.save("/kaggle/working/qwen_states.npy", np.load(FEATURES / "generation" / "triviaqa" / "features.npy", mmap_mode="r")[:N])
with open("/kaggle/working/qwen_records.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps({"id": r["id"], "input": r["input_text"], "generated_output": r["output_text"], "actual_output": r["ground_truth_answers"]}) + "\n")

In [ ]:
!hprobes predict --model /kaggle/working/model.pt --states /kaggle/working/qwen_states.npy --records /kaggle/working/qwen_records.jsonl --bleurt --device cuda --output /kaggle/working/qwen_predictions.csv

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 60, "display.width", 200)
q = pd.read_csv("/kaggle/working/qwen_predictions.csv"); g = pd.read_csv("/kaggle/working/gemma_predictions.csv")
orig = pd.read_csv(RUN_DIR / "results.csv").set_index("id")["sala_probability"]
ids = [r["id"] for r in rows]
print("Control: max |probability - original run| =", float((q["probability_correct"].values - orig.loc[ids].values).__abs__().max()))
show = ["input", "generated_output", "bleurt_score", "bleurt_label", "probability_correct", "verdict"]
print("\nQwen (control)"); display(q[show].round(3))
print("\nGemma-2-9B-it through the Qwen detector"); display(g[show].round(3))
print(f"Gemma answers correct by BLEURT: {int(g.bleurt_label.sum())}/{len(g)}; detector agrees with BLEURT on {int(g.detector_agrees_with_bleurt.sum())}/{len(g)}")